In [1]:
pip install transformers torch pandas scikit-learn tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install transformers torch pandas scikit-learn tqdm

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install scikit-learn
!pip install seaborn


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [5]:
import pandas as pd
import torch
from transformers import AutoTokenizer, BertForSequenceClassification, AutoConfig
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score
import multiprocessing
from transformers import (
    DebertaV2Tokenizer,
    DebertaV2ForSequenceClassification
)

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoConfig, BertForSequenceClassification
import multiprocessing
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    classification_report,
    accuracy_score
)
from tqdm import tqdm

class ModelPerformanceTracker:
    """
    A class to track and visualize model performance during training
    """
    def __init__(self):
        self.train_losses = []
        self.test_losses = []
        self.auroc_scores = []
        self.accuracies = []
        self.f1_scores = []

    def update(self, train_loss, test_loss, auroc, accuracy, f1_score):
        self.train_losses.append(train_loss)
        self.test_losses.append(test_loss)
        self.auroc_scores.append(auroc)
        self.accuracies.append(accuracy)
        self.f1_scores.append(f1_score)

    def plot_performance(self):
        """
        Create comprehensive visualization of model performance
        """
        plt.figure(figsize=(15, 10))
        
        # Losses
        plt.subplot(2, 2, 1)
        plt.plot(self.train_losses, label='Train Loss', color='blue')
        plt.plot(self.test_losses, label='Test Loss', color='red')
        plt.title('Loss Progression')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()

        # AUROC Scores
        plt.subplot(2, 2, 2)
        plt.plot(self.auroc_scores, label='AUROC', color='green')
        plt.title('ROC AUC Score')
        plt.xlabel('Epochs')
        plt.ylabel('AUROC')
        plt.legend()

        # Accuracy
        plt.subplot(2, 2, 3)
        plt.plot(self.accuracies, label='Accuracy', color='purple')
        plt.title('Model Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()

        # F1 Score
        plt.subplot(2, 2, 4)
        plt.plot(self.f1_scores, label='F1 Score', color='orange')
        plt.title('F1 Score')
        plt.xlabel('Epochs')
        plt.ylabel('F1 Score')
        plt.legend()

        plt.tight_layout()
        plt.savefig('model_performance_balanced.png')
        plt.close()

class EssaysDataset(Dataset):
    def __init__(self, essays_df, tokenizer, max_sequence_length):
        self.tokenizer = tokenizer
        self.max_sequence_length = max_sequence_length

        # Prepare the data
        self.texts = essays_df['text'].tolist()
        self.labels = essays_df['label'].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Tokenize the text
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_sequence_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def setup_data_loaders(train_essays, test_essays, tokenizer_name, batch_size, max_sequence_length):
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    # Create datasets
    train_dataset = EssaysDataset(train_essays, tokenizer, max_sequence_length)
    test_dataset = EssaysDataset(test_essays, tokenizer, max_sequence_length)

    # Create data loaders
    train_data_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        num_workers=multiprocessing.cpu_count(),
        shuffle=True,
        pin_memory=True
    )

    test_data_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        num_workers=multiprocessing.cpu_count(),
        shuffle=False,
        pin_memory=True
    )

    return train_data_loader, test_data_loader, tokenizer

def test_model(test_data_loader, model, device):
    model.eval()
    test_losses = []
    all_predictions = []
    all_actual_values = []
    all_raw_predictions = []

    with torch.no_grad():
        for batch in tqdm(test_data_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            test_losses.append(loss.item())

            # Get predictions (binary classification)
            batch_predictions = torch.argmax(logits, dim=1)
            batch_raw_predictions = torch.softmax(logits, dim=1)[:, 1]

            all_predictions.extend(batch_predictions.cpu().numpy())
            all_raw_predictions.extend(batch_raw_predictions.cpu().numpy())
            all_actual_values.extend(labels.cpu().numpy())

    # Compute evaluation metrics
    metrics = {
        'accuracy': accuracy_score(all_actual_values, all_predictions),
        'precision': precision_score(all_actual_values, all_predictions),
        'recall': recall_score(all_actual_values, all_predictions),
        'f1_score': f1_score(all_actual_values, all_predictions),
        'roc_auc': roc_auc_score(all_actual_values, all_raw_predictions),
        'avg_test_loss': np.mean(test_losses),
        'confusion_matrix': confusion_matrix(all_actual_values, all_predictions)
    }

    # Visualize Confusion Matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(metrics['confusion_matrix'], 
                annot=True, 
                fmt='d', 
                cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_on_balanced.png')
    plt.close()

    # Generate Detailed Classification Report
    print("\nDetailed Classification Report:")
    print(classification_report(
        all_actual_values, 
        all_predictions, 
        target_names=['Class 0', 'Class 1']
    ))

    # Print Individual Metrics
    print("\nEvaluation Metrics:")
    for metric, value in metrics.items():
        if metric != 'confusion_matrix':
            print(f"{metric}: {value}")
    
    return metrics

def train_model(train_essays, test_essays, learning_rate=2e-5, batch_size=16, epochs=3):
    # Hyperparameters
    MAX_SEQUENCE_LENGTH = 512
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    MODEL_NAME = "bert-base-uncased"

    # Setup data loaders
    train_data_loader, test_data_loader, tokenizer = setup_data_loaders(
        train_essays,
        test_essays,
        MODEL_NAME,
        batch_size,
        MAX_SEQUENCE_LENGTH
    )

    # Initialize performance tracker
    performance_tracker = ModelPerformanceTracker()

    # Initialize model with binary classification head
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
    torch.cuda.empty_cache()
    torch.set_float32_matmul_precision('medium')
    model = BertForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

    model = model.to(DEVICE)

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

    # Training loop
    best_auroc = 0
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0

        for batch in tqdm(train_data_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            # Move data to device
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            # Zero gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        # Calculate average train loss
        avg_train_loss = total_train_loss / len(train_data_loader)

        # Test the model after each epoch
        model_metrics = test_model(test_data_loader, model, DEVICE)

        # Update performance tracker
        performance_tracker.update(
            train_loss=avg_train_loss, 
            test_loss=model_metrics['avg_test_loss'],
            auroc=model_metrics['roc_auc'],
            accuracy=model_metrics['accuracy'],
            f1_score=model_metrics['f1_score']
        )

        # Print epoch summary
        print(f'Epoch {epoch+1}: '
              f'Train Loss = {avg_train_loss:.4f}, '
              f'Test Loss = {model_metrics["avg_test_loss"]:.4f}, '
              f'AUROC = {model_metrics["roc_auc"]:.4f}')

        # Update best AUROC
        best_auroc = max(best_auroc, model_metrics['roc_auc'])

    # Create performance visualization
    performance_tracker.plot_performance()

    return model, model_metrics

In [7]:
import re

# Text cleaning 
def clean_text(text):
    """
    Cleans raw text for BERT input.
    - Removes special characters.
    - Converts text to lowercase.
    - Removes extra whitespaces.
    """
    text = re.sub(r"[^a-zA-Z0-9\s]", "", str(text))  # Remove special characters
    text = text.lower()  # Convert to lowercase
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra whitespaces
    return text


In [8]:

def main():
    # Load data
    train_essays = pd.read_csv("reduced_train_balanced.csv")
    test_essays = pd.read_csv("reduced_test_balanced.csv")
    train_essays['text'] = train_essays['text'].apply(clean_text)
    test_essays['text'] = test_essays['text'].apply(clean_text)

    # Train the model
    model, model_metrics = train_model(train_essays, test_essays)

    # Save the model
    torch.save(model.state_dict(), 'bert_ai_text_detection_model_balanced.pth')

    

if __name__ == '__main__':
    main()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Evaluating: 100%|██████████| 193/193 [02:23<00:00,  1.34it/s]



Detailed Classification Report:
              precision    recall  f1-score   support

     Class 0       0.98      1.00      0.99      1537
     Class 1       1.00      0.98      0.99      1537

    accuracy                           0.99      3074
   macro avg       0.99      0.99      0.99      3074
weighted avg       0.99      0.99      0.99      3074


Evaluation Metrics:
accuracy: 0.9876382563435263
precision: 0.9986693280106453
recall: 0.9765777488614183
f1_score: 0.9875
roc_auc: 0.9989349673992505
avg_test_loss: 0.039907186960444395
Epoch 1: Train Loss = 0.1117, Test Loss = 0.0399, AUROC = 0.9989


Evaluating: 100%|██████████| 193/193 [02:24<00:00,  1.34it/s]



Detailed Classification Report:
              precision    recall  f1-score   support

     Class 0       0.99      0.99      0.99      1537
     Class 1       0.99      0.99      0.99      1537

    accuracy                           0.99      3074
   macro avg       0.99      0.99      0.99      3074
weighted avg       0.99      0.99      0.99      3074


Evaluation Metrics:
accuracy: 0.9886141834743006
precision: 0.9851421188630491
recall: 0.9921925829538061
f1_score: 0.9886547811993517
roc_auc: 0.9991411163962954
avg_test_loss: 0.03800613462226465
Epoch 2: Train Loss = 0.0275, Test Loss = 0.0380, AUROC = 0.9991


Evaluating: 100%|██████████| 193/193 [02:24<00:00,  1.34it/s]



Detailed Classification Report:
              precision    recall  f1-score   support

     Class 0       0.98      1.00      0.99      1537
     Class 1       1.00      0.98      0.99      1537

    accuracy                           0.99      3074
   macro avg       0.99      0.99      0.99      3074
weighted avg       0.99      0.99      0.99      3074


Evaluation Metrics:
accuracy: 0.9882888744307091
precision: 0.9993346640053227
recall: 0.9772283669486012
f1_score: 0.9881578947368421
roc_auc: 0.9995724630656768
avg_test_loss: 0.040945756977027126
Epoch 3: Train Loss = 0.0212, Test Loss = 0.0409, AUROC = 0.9996
